## IMPORTAR LIBRERIAS !!

In [10]:
import pandas as pd
import numpy as np


## En esta sección importamos los excel y limpiamos columnas basuras

In [11]:
# Importar datos
df_movimiento_contables = pd.read_excel('/home/datos/INOcost/BASE_MOVIMIENTOS_CONTABLES.xlsx')
df_relacion_costos = pd.read_excel('/home/datos/INOcost/TABLA RELACION AUXILIAR-TIPO DE COSTO.xlsx')

# Eliminar columnas basura que trae el Excel
cols_validas = [c for c in df_movimiento_contables.columns if not str(c).startswith('Unnamed') and c != '<']
df_movimiento_contables = df_movimiento_contables[cols_validas]

print(f"Movimientos:  {df_movimiento_contables.shape[0]:,} filas | {df_movimiento_contables.shape[1]} columnas")
print(f"Relación:     {df_relacion_costos.shape[0]:,} filas | {df_relacion_costos.shape[1]} columnas")

Movimientos:  93,468 filas | 16 columnas
Relación:     264 filas | 4 columnas


## En esta sección se pueden hacer varias limpiezas y transformaciones para asegurar que el mapeo entre auxiliares y tipos de costo sea lo más preciso posible.

In [ ]:

# Normalizar columnas clave: string limpio de 8 chars (mismo criterio que views.py)
df_movimiento_contables['auxiliar'] = (
    df_movimiento_contables['auxiliar'].astype(str).str.strip().str[:8]
)
df_relacion_costos['cuenta_auxiliar'] = (
    df_relacion_costos['cuenta_auxiliar'].astype(str).str.strip().str[:8]
)

# Diagnóstico de cobertura del mapeo
auxiliares_mov = set(df_movimiento_contables['auxiliar'].unique())
auxiliares_rel = set(df_relacion_costos['cuenta_auxiliar'].unique())
print(f"Auxiliares en movimientos:        {len(auxiliares_mov)}")
print(f"Auxiliares en tabla relación:     {len(auxiliares_rel)}")
print(f"Auxiliares CON mapeo:             {len(auxiliares_mov & auxiliares_rel)}")
print(f"Auxiliares SIN mapeo (sin match): {len(auxiliares_mov - auxiliares_rel)}")

Auxiliares en movimientos:        220
Auxiliares en tabla relación:     263
Auxiliares CON mapeo:             216
Auxiliares SIN mapeo (sin match): 4


## Mostrar auxiliares sin mapeo (sin match)

In [ ]:

df_sin_mapeo = df_movimiento_contables[~df_movimiento_contables['auxiliar'].isin(auxiliares_rel)]
print("\nEjemplos de auxiliares sin mapeo:")
print(df_sin_mapeo['auxiliar'].value_counts().head(10))


Ejemplos de auxiliares sin mapeo:
auxiliar
52020601    17
51140601     2
51080201     1
51130501     1
Name: count, dtype: int64


## Unimos ambos dataframes para asignar tipo_costo y detalle_costo a cada movimiento

In [ ]:
# Merge: LEFT JOIN movimientos ← relacion_costos por auxiliar
# left_on='auxiliar', right_on='cuenta_auxiliar'
# Se conservan TODOS los movimientos; los sin mapeo quedan con NaN en tipo/detalle_costo
df = df_movimiento_contables.merge(
    df_relacion_costos[['cuenta_auxiliar', 'tipo_costo', 'detalle_costo']],
    left_on='auxiliar',
    right_on='cuenta_auxiliar',
    how='left'
).drop(columns='cuenta_auxiliar')  # columna duplicada, ya tenemos 'auxiliar'

print(f"Filas totales:       {len(df):,}")
print(f"Con tipo_costo:      {df['tipo_costo'].notna().sum():,}")
print(f"Sin tipo_costo:      {df['tipo_costo'].isna().sum():,}")
df.head()

Filas totales:       93,474
Con tipo_costo:      93,453
Sin tipo_costo:      21


,clase,fecha,auxiliar,desc_auxiliar,c_o_mvto,desc_c_o_mvto,unidad_negocio_codigo,docto_proveedor,desc_unidad_negocio,tercero_mvto,razon_social,debito,credito,neto,tipo_doc_contable,grupo_contable,tipo_costo,detalle_costo
0,7 - COSTOS SGSSS,2025-09-26,73130601,ACUEDUCTO Y ALCANTARILLADO,3,SEDE CABECERA,1,3369294--08756797,SERVICIO MEDICO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,457880.0,0.0,457880.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
1,7 - COSTOS SGSSS,2025-09-26,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,9,3428394--08816506,INVESTIGACIÓN,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,22870.0,0.0,22870.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
2,7 - COSTOS SGSSS,2025-06-17,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,2,-07813286,CLINICA DE SUEÑO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,54300.0,0.0,54300.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
3,7 - COSTOS SGSSS,2025-02-14,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,2,-06485092,CLINICA DE SUEÑO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,120450.0,0.0,120450.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
4,7 - COSTOS SGSSS,2025-08-20,73130601,ACUEDUCTO Y ALCANTARILLADO,2,SEDE EL PRADO,4,3188591-08573928,PROGRAMAS INTEGRALES,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,455600.0,0.0,455600.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS


## Aquí se pueden aplicar reglas automáticas para asignar tipo/detalle de costo a auxiliares sin mapeo, basándose en patrones del número auxiliar.

In [ ]:
# Reglas automáticas para auxiliares sin mapeo según primer dígito
reglas_prefijo = {
    '4': ('INGRESO',    'INGRESO'),
    '5': ('GASTO',      'GASTO'),
    '6': ('COSTO',      'COSTO'),
}

mask_sin_mapeo = df['tipo_costo'].isna()

for prefijo, (tipo, detalle) in reglas_prefijo.items():
    mask = mask_sin_mapeo & df['auxiliar'].str.startswith(prefijo)
    df.loc[mask, 'tipo_costo']    = tipo
    df.loc[mask, 'detalle_costo'] = detalle

# Actualizar máscara: lo que sigue sin mapeo (casi todo será 7)
pendientes = df[df['tipo_costo'].isna()]['auxiliar'].unique()
print(f"Auxiliares aún sin mapeo ({len(pendientes)}) — requieren asignación manual:")
for a in sorted(pendientes):
    print(f"  {a}")

## ASIGNACIÓN MANUAL — completa el diccionario con los auxiliares listados arriba

In [ ]:

# Formato:  'auxiliar': ('tipo_costo', 'detalle_costo')
mapeo_manual = {
    # Ejemplo — reemplaza con los valores reales:
    # '73130601': ('CIF_SERV_PUBLICOS', 'CIF_SERV_PUBLICOS'),
    # '73200101': ('CIF_MANO_OBRA',     'CIF_MANO_OBRA'),
}

# Aplicar el mapeo manual
for auxiliar, (tipo, detalle) in mapeo_manual.items():
    mask = (df['auxiliar'] == auxiliar) & df['tipo_costo'].isna()
    df.loc[mask, 'tipo_costo']    = tipo
    df.loc[mask, 'detalle_costo'] = detalle

# Verificación final
sin_mapeo_final = df['tipo_costo'].isna().sum()
print(f"Filas sin mapeo después de asignación manual: {sin_mapeo_final}")
print(f"Cobertura total: {(~df['tipo_costo'].isna()).sum():,} / {len(df):,} filas")
df.head()

Filas sin mapeo después de asignación manual: 0
Cobertura total: 93,474 / 93,474 filas


,clase,fecha,auxiliar,desc_auxiliar,c_o_mvto,desc_c_o_mvto,unidad_negocio_codigo,docto_proveedor,desc_unidad_negocio,tercero_mvto,razon_social,debito,credito,neto,tipo_doc_contable,grupo_contable,tipo_costo,detalle_costo
0,7 - COSTOS SGSSS,2025-09-26,73130601,ACUEDUCTO Y ALCANTARILLADO,3,SEDE CABECERA,1,3369294--08756797,SERVICIO MEDICO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,457880.0,0.0,457880.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
1,7 - COSTOS SGSSS,2025-09-26,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,9,3428394--08816506,INVESTIGACIÓN,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,22870.0,0.0,22870.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
2,7 - COSTOS SGSSS,2025-06-17,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,2,-07813286,CLINICA DE SUEÑO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,54300.0,0.0,54300.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
3,7 - COSTOS SGSSS,2025-02-14,73130601,ACUEDUCTO Y ALCANTARILLADO,6,SEDE FOSUNAB,2,-06485092,CLINICA DE SUEÑO,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,120450.0,0.0,120450.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS
4,7 - COSTOS SGSSS,2025-08-20,73130601,ACUEDUCTO Y ALCANTARILLADO,2,SEDE EL PRADO,4,3188591-08573928,PROGRAMAS INTEGRALES,890200162,ACUEDUCTO METROPOLITANO DE BUCARAMANGA SA ESP,455600.0,0.0,455600.0,Factura de servicio compra,73 - COSTOS INDIRECTOS,CIF,CIF SERVICIOS PUBLICOS


In [19]:
# Tabla filtrada: solo CIF, sin Cúcuta
tabla_cif = (
    df[
        (df['tipo_costo'] == 'CIF') &
        (~df['desc_c_o_mvto'].str.contains('cucuta', case=False, na=False))
    ]
    [['desc_c_o_mvto', 'desc_unidad_negocio', 'detalle_costo', 'neto']]
    .copy()
)

print(f"Filas: {len(tabla_cif):,}")
print(f"Total neto: {tabla_cif['neto'].sum():,.2f}")
tabla_cif

Filas: 8,359
Total neto: 4,272,122,746.66


,desc_c_o_mvto,desc_unidad_negocio,detalle_costo,neto
0,SEDE CABECERA,SERVICIO MEDICO,CIF SERVICIOS PUBLICOS,457880.00
1,SEDE FOSUNAB,INVESTIGACIÓN,CIF SERVICIOS PUBLICOS,22870.00
2,SEDE FOSUNAB,CLINICA DE SUEÑO,CIF SERVICIOS PUBLICOS,54300.00
3,SEDE FOSUNAB,CLINICA DE SUEÑO,CIF SERVICIOS PUBLICOS,120450.00
4,SEDE EL PRADO,PROGRAMAS INTEGRALES,CIF SERVICIOS PUBLICOS,455600.00
...,...,...,...,...
81615,SEDE FOSUNAB,SERVICIO MEDICO,"OTROS CIF UTILES, PAPELERIA Y FOTOCOPIAS",392380.23
81616,SEDE PRINCIPAL,CLINICA DE SUEÑO,"OTROS CIF UTILES, PAPELERIA Y FOTOCOPIAS",49857.42
81617,SEDE CABECERA,ADMINISTRACION (BACKOFFICE),"OTROS CIF UTILES, PAPELERIA Y FOTOCOPIAS",-16100.00
81618,SEDE FOSUNAB,LABORATORIO PULMONAR,"OTROS CIF UTILES, PAPELERIA Y FOTOCOPIAS",646618.03


In [20]:
# Exportar a Excel para análisis en Power BI
tabla_cif.to_excel('/home/datos/INOcost/tabla_cif.xlsx', index=False)